<a href="https://colab.research.google.com/github/mhmoud456/Drought-Risk-Prediction-and-Classification/blob/main/DroughtRiskPredictionandClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import pandas as pd

def fetch_drought_climate_data(lat=33.5138, lon=36.2765, start_year=2010, end_year=2025):
    """
    جلب بيانات الأرصاد الجوية الشهرية (درجات الحرارة وهطول الأمطار) من NASA POWER API
    الإحداثيات الافتراضية هنا لمدينة دمشق (يمكن تعديلها حسب حقل الدراسة)
    """
    url = f"https://power.larc.nasa.gov/api/temporal/monthly/point?parameters=T2M,T2M_MAX,PRECTOTCORR&community=AG&longitude={lon}&latitude={lat}&start={start_year}&end={end_year}&format=JSON"

    print("⏳ جاري الاتصال بـ NASA POWER API لجلب البيانات المناخية...")
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        parameters = data['properties']['parameter']

        # استخراج المتغيرات المطلوبة
        t2m = parameters['T2M']              # متوسط درجات الحرارة
        t2m_max = parameters['T2M_MAX']      # درجات الحرارة القصوى
        prectot = parameters['PRECTOTCORR']  # هطول الأمطار الصحيح

        # تحويلها إلى جدول DataFrame
        df = pd.DataFrame({
            'T2M': pd.Series(t2m),
            'T2M_MAX': pd.Series(t2m_max),
            'Monthly_Precipitation': pd.Series(prectot)
        })

        df.index.name = 'YearMonth'
        df.reset_index(inplace=True)

        print("✅ تم سحب البيانات المناخية بنجاح!")
        return df
    else:
        print(f"❌ حدث خطأ في الاتصال: {response.status_code}")
        return None

# تنفيذ الدالة وجلب البيانات
df_climate = fetch_drought_climate_data()

# عرض أول 12 صفاً من البيانات المسحوبة للتأكد
if df_climate is not None:
    print(df_climate.head(12))

⏳ جاري الاتصال بـ NASA POWER API لجلب البيانات المناخية...
✅ تم سحب البيانات المناخية بنجاح!
   YearMonth    T2M  T2M_MAX  Monthly_Precipitation
0     201001   7.70    20.36                   2.52
1     201002   7.68    21.90                   1.49
2     201003  11.37    27.53                   0.18
3     201004  14.67    27.63                   0.07
4     201005  19.91    33.72                   0.00
5     201006  23.49    36.63                   0.05
6     201007  27.08    41.89                   0.00
7     201008  29.03    41.48                   0.00
8     201009  24.51    35.40                   0.03
9     201010  19.84    33.54                   0.35
10    201011  15.83    26.80                   0.00
11    201012   9.02    23.43                   2.58


In [3]:
# 1. فحص معلومات الجدول العامة والتحقق من وجود قيم مفقودة (Missing Values)
print("🔍 معلومات عامة عن جدول البيانات المسحوبة:")
print(df_climate.info())

print("\n" + "="*40)
print("📊 عدد القيم المفقودة (NaN) في كل عمود:")
print("="*40)
print(df_climate.isnull().sum())

# 2. ترتيب البيانات زمنياً للتأكد من تسلسل الأشهر بشكل صحيح
df_climate = df_climate.sort_values('YearMonth').copy()

# 3. معالجة القيم المفقودة بطريقة سلاسل الزمن (Time-Series Imputation)
# نستخدم forward fill ثم backward fill لضمان عدم قطع التسلسل الزمني
df_climate['T2M'] = df_climate['T2M'].ffill().bfill()
df_climate['T2M_MAX'] = df_climate['T2M_MAX'].ffill().bfill()
df_climate['Monthly_Precipitation'] = df_climate['Monthly_Precipitation'].ffill().bfill()

print("\n" + "="*40)
print("✅ تم فحص البيانات ومعالجة أي نقص محتمل بنجاح وأصبحت جاهزة للخطوة التالية!")
print("="*40)
print(df_climate.head())

🔍 معلومات عامة عن جدول البيانات المسحوبة:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   YearMonth              208 non-null    object 
 1   T2M                    208 non-null    float64
 2   T2M_MAX                208 non-null    float64
 3   Monthly_Precipitation  208 non-null    float64
dtypes: float64(3), object(1)
memory usage: 6.6+ KB
None

📊 عدد القيم المفقودة (NaN) في كل عمود:
YearMonth                0
T2M                      0
T2M_MAX                  0
Monthly_Precipitation    0
dtype: int64

✅ تم فحص البيانات ومعالجة أي نقص محتمل بنجاح وأصبحت جاهزة للخطوة التالية!
  YearMonth    T2M  T2M_MAX  Monthly_Precipitation
0    201001   7.70    20.36                   2.52
1    201002   7.68    21.90                   1.49
2    201003  11.37    27.53                   0.18
3    201004  14.67    27.63                  

In [4]:
import numpy as np

# نسخ الجدول لنعمل عليه بأمان
df_features = df_climate.copy()

print("⏳ جاري هندسة الميزات الزمنية (Lagged & Rolling Features)...")

# 1. ميزات تباطؤ الأمطار (Lagged Features)
# Lag_1: أمطار الشهر الماضي
# Lag_2: أمطار قبل شهرين
df_features['Precip_Lag_1'] = df_features['Monthly_Precipitation'].shift(1)
df_features['Precip_Lag_2'] = df_features['Monthly_Precipitation'].shift(2)

# 2. ميزات المتوسطات التراكمية للأمطار (Rolling Features)
# متوسط الأمطار خلال آخر 3 أشهر وآخر 6 أشهر
df_features['Precip_Rolling_3M'] = df_features['Monthly_Precipitation'].rolling(window=3).mean()
df_features['Precip_Rolling_6M'] = df_features['Monthly_Precipitation'].rolling(window=6).mean()

# 3. ميزات درجات الحرارة التراكمية (الضغط الحراري)
# متوسط الحرارة ومتوسط الحرارة القصوى لآخر 3 أشهر
df_features['Temp_Rolling_3M'] = df_features['T2M'].rolling(window=3).mean()
df_features['Max_Temp_Rolling_3M'] = df_features['T2M_MAX'].rolling(window=3).mean()

# 4. حساب الشذوذ المناخي للحرارة (Temp Anomaly)
# مقياس لمدى انحراف درجة الحرارة عن المعدل التاريخي العام
temp_mean = df_features['T2M'].mean()
temp_std = df_features['T2M'].std()
df_features['Temp_Anomaly'] = (df_features['T2M'] - temp_mean) / (temp_std + 1e-8)

# 5. تنظيف أي صفوف فارغة (NaN) تنتج في البداية بسبب عمليات الـ Shift والـ Rolling
df_features.dropna(inplace=True)

print("✅ تم هندسة جميع الميزات الزمنية بنجاح!")
print(f"📊 عدد الأسطر بعد تصفية الصفوف الأولى الفارغة: {len(df_features)} شهر")

# عرض عينة من الأعمدة الجديدة للتأكد
print("\n🔍 عينة من الجدول بعد إضافة الميزات:")
print(df_features[['YearMonth', 'Monthly_Precipitation', 'Precip_Lag_1', 'Precip_Rolling_3M', 'Temp_Anomaly']].head())

⏳ جاري هندسة الميزات الزمنية (Lagged & Rolling Features)...
✅ تم هندسة جميع الميزات الزمنية بنجاح!
📊 عدد الأسطر بعد تصفية الصفوف الأولى الفارغة: 203 شهر

🔍 عينة من الجدول بعد إضافة الميزات:
  YearMonth  Monthly_Precipitation  Precip_Lag_1  Precip_Rolling_3M  \
5    201006                   0.05          0.00           0.040000   
6    201007                   0.00          0.05           0.016667   
7    201008                   0.00          0.00           0.016667   
8    201009                   0.03          0.00           0.010000   
9    201010                   0.35          0.03           0.126667   

   Temp_Anomaly  
5      0.963868  
6      1.438486  
7      1.696287  
8      1.098718  
9      0.481317  


In [10]:
import numpy as np

# نسخ الجدول لنعمل عليه بأمان
df_target = df_features.copy()

print("⏳ جاري حساب عتبة الجفاف وتصنيف الأشهر...")

# 1. حساب عتبة الـ 25 الأدنى (25th Percentile) لهطولات الأمطار التاريخية
drought_threshold = np.percentile(df_target['Monthly_Precipitation'], 25)

print(f"📊 عتبة الجفاف المحسوبة تاريخياً: {drought_threshold:.2f} mm")

# 2. إنشاء العمود المستهدف (Drought_Target)
df_target['Drought_Target'] = np.where(df_target['Monthly_Precipitation'] <= drought_threshold, 1, 0)

# 3. طباعة إحصائيات سريعة للتأكد من توزيع الحالات
total_months = len(df_target)
drought_months = df_target['Drought_Target'].sum()
normal_months = total_months - drought_months

print("\n" + "="*40)
print("📈 إحصائيات تصنيف الأشهر:")
print("="*40)
print(f"📅 إجمالي عدد الأشهر في الدراسة: {total_months} شهر")
print(f"☀️ عدد الأشهر الطبيعية (0): {normal_months} شهر")
print(f"🌵 عدد الأشهر الجافة (1): {drought_months} شهر")
print(f"📌 نسبة الأشهر الجافة الفعلية: {(drought_months/total_months)*100:.2f}%")
print("="*40)

# عرض عينة للتأكد من ظهور عمود الهدف الجديد
print("\n🔍 عينة من الجدول مع عمود الهدف (Drought_Target):")
print(df_target[['YearMonth', 'Monthly_Precipitation', 'Precip_Lag_1', 'Drought_Target']].head(10))

⏳ جاري حساب عتبة الجفاف وتصنيف الأشهر...
📊 عتبة الجفاف المحسوبة تاريخياً: 0.01 mm

📈 إحصائيات تصنيف الأشهر:
📅 إجمالي عدد الأشهر في الدراسة: 203 شهر
☀️ عدد الأشهر الطبيعية (0): 151 شهر
🌵 عدد الأشهر الجافة (1): 52 شهر
📌 نسبة الأشهر الجافة الفعلية: 25.62%

🔍 عينة من الجدول مع عمود الهدف (Drought_Target):
   YearMonth  Monthly_Precipitation  Precip_Lag_1  Drought_Target
5     201006                   0.05          0.00               0
6     201007                   0.00          0.05               1
7     201008                   0.00          0.00               1
8     201009                   0.03          0.00               0
9     201010                   0.35          0.03               0
10    201011                   0.00          0.35               1
11    201012                   2.58          0.00               0
12    201013                   0.60          2.58               0
13    201101                   1.44          0.60               0
14    201102                   3.02  

In [11]:
import pandas as pd

# 1. تحديد أعمدة الميزات (Features X) الخالية تماماً من تسريب البيانات
# نحن نعتمد فقط على ميزات الـ Lag، الـ Rolling، والحرارة، ولا نضع Monthly_Precipitation الحالي أبداً
feature_columns = [
    'Precip_Lag_1', 'Precip_Lag_2',
    'Precip_Rolling_3M', 'Precip_Rolling_6M',
    'T2M', 'T2M_MAX',
    'Temp_Rolling_3M', 'Max_Temp_Rolling_3M',
    'Temp_Anomaly'
]

# مصفوفة المدخلات المستقلة (X) والعمود المستهدف (y)
X = df_target[feature_columns]
y = df_target['Drought_Target']

# 2. التقسيم الزمني الأكاديمي الصارم (Chronological Split)
# حساب مؤشر نقطة الفصل (80% تدريب و 20% اختبار)
split_index = int(len(X) * 0.8)

# تقسيم مصفوفة الميزات والهدف زمنياً
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# 3. طباعة نتائج التقسيم للتأكد من جاهزية المصفوفات
print("="*60)
print("✂️ تم تجهيز وتقسيم البيانات زمنياً بنجاح تام:")
print("="*60)
print(f"📊 حجم بيانات التدريب (X_train): {X_train.shape[0]} شهر (تُمثل السنوات الأولى للتعلم)")
print(f"🧪 حجم بيانات الاختبار (X_test): {X_test.shape[0]} شهر (تُمثل السنوات الأخيرة للاختبار المجهول)")
print(f"📌 عدد ميزات المدخلات الزمنية المستخدمة: {X_train.shape[1]} ميزات")
print("="*60)

# عرض عينة صغيرة من بداية بيانات الاختبار للتأكد
print("\n🔍 عينة من موعد بدء بيانات الاختبار (X_test):")
print(X_test.head(3))

✂️ تم تجهيز وتقسيم البيانات زمنياً بنجاح تام:
📊 حجم بيانات التدريب (X_train): 162 شهر (تُمثل السنوات الأولى للتعلم)
🧪 حجم بيانات الاختبار (X_test): 41 شهر (تُمثل السنوات الأخيرة للاختبار المجهول)
📌 عدد ميزات المدخلات الزمنية المستخدمة: 9 ميزات

🔍 عينة من موعد بدء بيانات الاختبار (X_test):
     Precip_Lag_1  Precip_Lag_2  Precip_Rolling_3M  Precip_Rolling_6M    T2M  \
167          1.04          0.17           0.526667           0.263333   8.29   
168          0.37          1.04           0.670000           0.363333  16.14   
169          0.60          0.37           0.543333           0.473333   6.46   

     T2M_MAX  Temp_Rolling_3M  Max_Temp_Rolling_3M  Temp_Anomaly  
167    17.99        13.246667            25.096667     -1.045658  
168    40.95        12.246667            27.440000     -0.007843  
169    17.96        10.296667            25.633333     -1.287594  


In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("⏳ جاري تدريب نموذج شجرة القرار (Decision Tree)...")

# 1. إنشاء النموذج مع تثبيت random_state لضمان ثبات النتائج
dt_model = DecisionTreeClassifier(random_state=42, max_depth=4)

# 2. تدريب النموذج على بيانات التدريب
dt_model.fit(X_train, y_train)

# 3. التنبؤ على بيانات الاختبار (المستقبلية المجهولة)
dt_pred = dt_model.predict(X_test)

# 4. حساب مقاييس الأداء الأكاديمية
dt_acc = accuracy_score(y_test, dt_pred)
dt_prec = precision_score(y_test, dt_pred, zero_division=0)
dt_rec = recall_score(y_test, dt_pred, zero_division=0)
dt_f1 = f1_score(y_test, dt_pred, zero_division=0)
dt_cm = confusion_matrix(y_test, dt_pred)

print("\n" + "="*50)
print("📊 تقرير أداء نموذج شجرة القرار (Decision Tree):")
print("="*50)
print(f"🎯 الدقة العامة (Accuracy): {dt_acc * 100:.2f}%")
print(f"🔍 دقة رصد الجفاف (Precision): {dt_prec * 100:.2f}%")
print(f"🚨 معدل الاسترجاع (Recall - عدم تفويت الجفاف): {dt_rec * 100:.2f}%")
print(f"⚖️ النتيجة التوافقية (F1-Score): {dt_f1:.4f}")
print("\nمصفوفة الارتباك (Confusion Matrix):")
print(dt_cm)
print("="*50)

⏳ جاري تدريب نموذج شجرة القرار (Decision Tree)...

📊 تقرير أداء نموذج شجرة القرار (Decision Tree):
🎯 الدقة العامة (Accuracy): 85.37%
🔍 دقة رصد الجفاف (Precision): 66.67%
🚨 معدل الاسترجاع (Recall - عدم تفويت الجفاف): 80.00%
⚖️ النتيجة التوافقية (F1-Score): 0.7273

مصفوفة الارتباك (Confusion Matrix):
[[27  4]
 [ 2  8]]


In [13]:
from sklearn.ensemble import RandomForestClassifier

print("⏳ جاري تدريب نموذج الغابات العشوائية (Random Forest)...")

# 1. إنشاء النموذج بتجميع 100 شجرة قرار
rf_model = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)

# 2. التدريب والتنبؤ
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# 3. حساب المقاييس
rf_acc = accuracy_score(y_test, rf_pred)
rf_prec = precision_score(y_test, rf_pred, zero_division=0)
rf_rec = recall_score(y_test, rf_pred, zero_division=0)
rf_f1 = f1_score(y_test, rf_pred, zero_division=0)
rf_cm = confusion_matrix(y_test, rf_pred)

print("\n" + "="*50)
print("📊 تقرير أداء نموذج الغابات العشوائية (Random Forest):")
print("="*50)
print(f"🎯 الدقة العامة (Accuracy): {rf_acc * 100:.2f}%")
print(f"🔍 دقة رصد الجفاف (Precision): {rf_prec * 100:.2f}%")
print(f"🚨 معدل الاسترجاع (Recall - عدم تفويت الجفاف): {rf_rec * 100:.2f}%")
print(f"⚖️ النتيجة التوافقية (F1-Score): {rf_f1:.4f}")
print("\nمصفوفة الارتباك (Confusion Matrix):")
print(rf_cm)
print("="*50)

⏳ جاري تدريب نموذج الغابات العشوائية (Random Forest)...

📊 تقرير أداء نموذج الغابات العشوائية (Random Forest):
🎯 الدقة العامة (Accuracy): 97.56%
🔍 دقة رصد الجفاف (Precision): 90.91%
🚨 معدل الاسترجاع (Recall - عدم تفويت الجفاف): 100.00%
⚖️ النتيجة التوافقية (F1-Score): 0.9524

مصفوفة الارتباك (Confusion Matrix):
[[30  1]
 [ 0 10]]


In [14]:
from xgboost import XGBClassifier

print("⏳ جاري تدريب نموذج XGBoost المتقدم...")

# 1. إنشاء نموذج التعزيز التدرجي
xgb_model = XGBClassifier(random_state=42, n_estimators=100, max_depth=3, learning_rate=0.03)

# 2. التدريب والتنبؤ
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

# 3. حساب المقاييس
xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_prec = precision_score(y_test, xgb_pred, zero_division=0)
xgb_rec = recall_score(y_test, xgb_pred, zero_division=0)
xgb_f1 = f1_score(y_test, xgb_pred, zero_division=0)
xgb_cm = confusion_matrix(y_test, xgb_pred)

print("\n" + "="*50)
print("📊 تقرير أداء نموذج XGBoost:")
print("="*50)
print(f"🎯 الدقة العامة (Accuracy): {xgb_acc * 100:.2f}%")
print(f"🔍 دقة رصد الجفاف (Precision): {xgb_prec * 100:.2f}%")
print(f"🚨 معدل الاسترجاع (Recall - عدم تفويت الجفاف): {xgb_rec * 100:.2f}%")
print(f"⚖️ النتيجة التوافقية (F1-Score): {xgb_f1:.4f}")
print("\nمصفوفة الارتباك (Confusion Matrix):")
print(xgb_cm)
print("="*50)

⏳ جاري تدريب نموذج XGBoost المتقدم...

📊 تقرير أداء نموذج XGBoost:
🎯 الدقة العامة (Accuracy): 95.12%
🔍 دقة رصد الجفاف (Precision): 83.33%
🚨 معدل الاسترجاع (Recall - عدم تفويت الجفاف): 100.00%
⚖️ النتيجة التوافقية (F1-Score): 0.9091

مصفوفة الارتباك (Confusion Matrix):
[[29  2]
 [ 0 10]]
